# Notebook 06 — White-Box Membership Inference Attack
**Black-Box MIA in Federated Learning | MSc Thesis**

This notebook implements a **white-box membership inference attack** based on
Nasr et al. (2019) *"Comprehensive Privacy Analysis of Deep Learning"*.

## What this notebook does
- Trains the target FL model on CIFAR-10 (same setup as all previous notebooks)
- Implements white-box gradient feature extraction using TensorFlow GradientTape
- Trains and evaluates the white-box attack classifier (RQ7)
- Compares white-box against loss-based and shadow model attacks
- Evaluates under both IID and Non-IID conditions


In [ ]:
# ── 1. Imports & Configuration ────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score, precision_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Configuration (identical to all previous notebooks) ───────────────────────
RANDOM_SEED   = 42
NUM_CLIENTS   = 5
NUM_ROUNDS    = 10
LOCAL_EPOCHS  = 3
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
NUM_CLASSES   = 10

# White-box attack configuration
N_ATTACK      = 2000   # per class (members + non-members)
                       # kept smaller because gradient computation is slow per-sample

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Configuration ready.')

In [ ]:
# ── 2. Load CIFAR-10 ──────────────────────────────────────────────────────────
print('Loading CIFAR-10...')
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()
print(f'Train: {x_train.shape}, Test: {x_test.shape}')

In [ ]:
# ── 3. Helper Functions ───────────────────────────────────────────────────────

def partition_data_iid(x, y, num_clients, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(x))
    splits = np.array_split(indices, num_clients)
    client_data    = [(x[idx], y[idx]) for idx in splits]
    client_indices = [idx.tolist() for idx in splits]
    return client_data, client_indices


def partition_data_noniid(x, y, num_clients, classes_per_client, seed=42):
    rng = np.random.default_rng(seed)
    num_classes = len(np.unique(y))
    class_indices = {c: np.where(y == c)[0] for c in range(num_classes)}
    for c in class_indices:
        class_indices[c] = rng.permutation(class_indices[c])
    client_indices = [[] for _ in range(num_clients)]
    for i in range(num_clients):
        assigned = [(i * classes_per_client + j) % num_classes
                    for j in range(classes_per_client)]
        for c in assigned:
            client_indices[i].extend(class_indices[c].tolist())
    client_data = [(x[idx], y[idx]) for idx in client_indices]
    return client_data, client_indices


def build_model(num_classes=10, learning_rate=0.001):
    """Small CNN — same architecture as all previous notebooks."""
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def fedavg_aggregate(client_weights, client_sizes):
    total = sum(client_sizes)
    return [
        sum(w * (n / total) for w, n in zip(layer_ws, client_sizes))
        for layer_ws in zip(*client_weights)
    ]


def local_train(global_weights, x_client, y_client,
                local_epochs, batch_size, lr):
    model = build_model(learning_rate=lr)
    model.set_weights(global_weights)
    model.fit(x_client, y_client,
              epochs=local_epochs, batch_size=batch_size, verbose=0)
    return model.get_weights()


def run_fedavg(client_data, client_sizes, num_rounds, local_epochs,
               batch_size, lr, x_val, y_val, label=''):
    global_model   = build_model(learning_rate=lr)
    global_weights = global_model.get_weights()
    round_accs = []
    for r in range(1, num_rounds + 1):
        cw_list = [
            local_train(global_weights, xc, yc, local_epochs, batch_size, lr)
            for xc, yc in client_data
        ]
        global_weights = fedavg_aggregate(cw_list, client_sizes)
        global_model.set_weights(global_weights)
        _, acc = global_model.evaluate(x_val, y_val, verbose=0)
        round_accs.append(acc)
        print(f'  {label} Round {r:2d}/{num_rounds} — acc: {acc:.4f}')
    return global_model, round_accs


def evaluate_attack(clf, X, y):
    y_scores = clf.predict_proba(X)[:, 1]
    y_pred   = clf.predict(X)
    return {
        'auc':       roc_auc_score(y, y_scores),
        'accuracy':  accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall':    recall_score(y, y_pred, zero_division=0),
        'y_scores':  y_scores,
        'y_true':    y,
    }


def loss_based_auc(model, x_members, y_members, x_nonmembers, y_nonmembers):
    """Compute loss-based attack AUC (lower loss = member)."""
    loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction='none')
    m_loss  = loss_fn(y_members,    model(x_members,    training=False)).numpy()
    nm_loss = loss_fn(y_nonmembers, model(x_nonmembers, training=False)).numpy()
    scores  = np.concatenate([-m_loss, -nm_loss])
    labels  = np.array([1]*len(m_loss) + [0]*len(nm_loss))
    return roc_auc_score(labels, scores)


print('Helper functions defined.')

## Core: White-Box Gradient Feature Extraction

This is the key function. For each sample, it uses TensorFlow's `GradientTape`
to compute the gradient of the loss with respect to every layer's weights,
then extracts the L2 norm of each layer's gradient as a feature.

Members → small gradient norms (model has already learned them)  
Non-members → large gradient norms (model would need to change to fit them)

In [ ]:
# ── 4. White-Box Gradient Feature Extraction ──────────────────────────────────

loss_fn = keras.losses.SparseCategoricalCrossentropy()

@tf.function
def compute_gradients(model, x_single, y_single):
    """
    Compute gradient of loss w.r.t. all trainable weights for one sample.
    Returns list of gradient tensors, one per trainable weight matrix.
    """
    with tf.GradientTape() as tape:
        pred = model(x_single, training=False)
        loss = loss_fn(y_single, pred)
    grads = tape.gradient(loss, model.trainable_variables)
    return grads, loss, pred


def extract_whitebox_features(model, x_samples, y_samples, batch_size=50, label=''):
    """
    Extract white-box features for a set of samples.
    
    Feature vector per sample:
    - L2 norm of gradient for each trainable layer  (captures per-layer sensitivity)
    - Total gradient L2 norm                         (overall gradient magnitude)
    - Output layer gradient norm specifically         (most directly linked to prediction)
    - Loss value                                      (same signal as loss-based attack)
    - Max softmax confidence                          (same signal as threshold attack)
    - Prediction entropy                              (uncertainty of the model)
    """
    n = len(x_samples)
    all_features = []

    for i in range(0, n, batch_size):
        if i % 500 == 0:
            print(f'  {label} Processing samples {i}/{n}...')

        batch_x = x_samples[i:i+batch_size]
        batch_y = y_samples[i:i+batch_size]

        for j in range(len(batch_x)):
            x_s = tf.expand_dims(batch_x[j], 0)    # shape (1, 32, 32, 3)
            y_s = tf.expand_dims(batch_y[j], 0)    # shape (1,)

            grads, loss_val, pred = compute_gradients(model, x_s, y_s)

            # Per-layer gradient L2 norms
            layer_norms = [tf.norm(g).numpy() for g in grads if g is not None]

            # Total gradient norm (sum across all layers)
            total_norm = float(np.sum(layer_norms))

            # Output layer gradient norm (last trainable variable = output Dense bias)
            output_norm = layer_norms[-1] if layer_norms else 0.0

            # Loss value
            loss_val = float(loss_val.numpy())

            # Softmax features
            probs      = pred.numpy()[0]            # shape (10,)
            max_conf   = float(probs.max())
            entropy    = float(-np.sum(probs * np.log(probs + 1e-10)))

            # Build feature vector:
            # [layer_norms..., total_norm, output_norm, loss, max_conf, entropy]
            feat = layer_norms + [total_norm, output_norm, loss_val, max_conf, entropy]
            all_features.append(feat)

    return np.array(all_features)


print('White-box feature extractor defined.')
print()
print('Features extracted per sample:')
print('  - L2 norm of gradient for each trainable weight matrix')
print('  - Total gradient L2 norm (sum across all layers)')
print('  - Output layer gradient norm')
print('  - Loss value (cross-entropy)')
print('  - Max softmax confidence')
print('  - Prediction entropy')

## Step 1 — Train Target FL Model (IID)

In [ ]:
# ── Step 1: Train Target FL Model (CIFAR-10, IID) ────────────────────────────
print('Step 1: Training target FL model on CIFAR-10 (IID, 5 clients, 10 rounds)...')

client_data, client_indices = partition_data_iid(
    x_train, y_train, NUM_CLIENTS, RANDOM_SEED
)
client_sizes   = [len(xc) for xc, _ in client_data]
member_indices = [i for sub in client_indices for i in sub]

global_model, round_accs = run_fedavg(
    client_data, client_sizes,
    NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE, LEARNING_RATE,
    x_test, y_test, label='[IID]'
)

global_acc_iid = round_accs[-1]
print(f'\nTarget FL model final accuracy: {global_acc_iid:.4f}')
print(f'Number of trainable weight matrices: {len(global_model.trainable_variables)}')

## Step 2 — Extract White-Box Features

This is the computationally expensive step. For each sample we run a forward pass
AND a backward pass (gradient computation) through the model.
This takes longer than the black-box attack but gives much richer features.

In [ ]:
# ── Step 2: Extract White-Box Features for IID Attack ────────────────────────
print('Step 2: Extracting white-box gradient features...')
print(f'Computing gradients for {N_ATTACK} members + {N_ATTACK} non-members')
print('This takes longer than black-box (one forward+backward pass per sample)\n')

rng = np.random.default_rng(RANDOM_SEED)

# Sample members and non-members
m_idx  = rng.choice(member_indices, size=N_ATTACK, replace=False)
nm_idx = rng.choice(len(x_test),    size=N_ATTACK, replace=False)

x_members     = x_train[m_idx]
y_members     = y_train[m_idx]
x_nonmembers  = x_test[nm_idx]
y_nonmembers  = y_test[nm_idx]

print('Extracting features for MEMBERS...')
feat_members = extract_whitebox_features(
    global_model, x_members, y_members, label='Members'
)

print('\nExtracting features for NON-MEMBERS...')
feat_nonmembers = extract_whitebox_features(
    global_model, x_nonmembers, y_nonmembers, label='Non-members'
)

print(f'\nFeature matrix shape (members):     {feat_members.shape}')
print(f'Feature matrix shape (non-members): {feat_nonmembers.shape}')
print(f'Features per sample: {feat_members.shape[1]}')

# Build attack dataset
X_attack = np.vstack([feat_members, feat_nonmembers])
y_attack = np.array([1]*N_ATTACK + [0]*N_ATTACK)

print(f'\nFull attack dataset: {X_attack.shape}')

In [ ]:
# ── Step 3: Train & Evaluate White-Box Attack Classifier (IID) ───────────────
print('Step 3: Training white-box attack classifier...')

# Scale features — gradient norms can have very different magnitudes across layers
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_attack)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_attack,
    test_size=0.3, stratify=y_attack, random_state=RANDOM_SEED
)

wb_clf_iid = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
wb_clf_iid.fit(X_tr, y_tr)

wb_metrics_iid = evaluate_attack(wb_clf_iid, X_te, y_te)

# Loss-based AUC as reference
loss_auc_iid = loss_based_auc(
    global_model, x_members, y_members, x_nonmembers, y_nonmembers
)

# Unrealistic black-box LR baseline for comparison
probs_m  = global_model.predict(x_members,    batch_size=256, verbose=0)
probs_nm = global_model.predict(x_nonmembers, batch_size=256, verbose=0)
X_bb     = np.vstack([probs_m, probs_nm])
X_bb_tr, X_bb_te, y_bb_tr, y_bb_te = train_test_split(
    X_bb, y_attack, test_size=0.3, stratify=y_attack, random_state=RANDOM_SEED
)
bb_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
bb_clf.fit(X_bb_tr, y_bb_tr)
bb_metrics_iid = evaluate_attack(bb_clf, X_bb_te, y_bb_te)

print()
print('═' * 65)
print('WHITE-BOX ATTACK RESULTS — CIFAR-10 (IID, 5 clients)')
print('═' * 65)
print(f'  Target model accuracy          : {global_acc_iid:.4f}')
print()
print(f'  {"Method":<40} {"AUC":>8} {"Recall":>8}')
print(f'  {"-"*56}')
print(f'  {"Black-box LR (unrealistic)":<40} {bb_metrics_iid["auc"]:>8.4f} {bb_metrics_iid["recall"]:>8.4f}')
print(f'  {"Loss-based (no training)":<40} {loss_auc_iid:>8.4f} {"N/A":>8}')
print(f'  {"White-box (gradient features)":<40} {wb_metrics_iid["auc"]:>8.4f} {wb_metrics_iid["recall"]:>8.4f}')
print(f'  {"Random chance":<40} {0.5000:>8.4f} {0.5000:>8.4f}')
print('═' * 65)

## Step 4 — White-Box Attack Under Non-IID

In [ ]:
# ── Step 4: Train Non-IID Target Model ───────────────────────────────────────
print('Step 4: Training Non-IID target FL model (5 clients, 5/10 classes each)...')

noniid_client_data, noniid_client_indices = partition_data_noniid(
    x_train, y_train,
    num_clients=NUM_CLIENTS,
    classes_per_client=5,
    seed=RANDOM_SEED
)
noniid_client_sizes   = [len(xc) for xc, _ in noniid_client_data]
noniid_member_indices = [i for sub in noniid_client_indices for i in sub]

noniid_model, noniid_round_accs = run_fedavg(
    noniid_client_data, noniid_client_sizes,
    NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE, LEARNING_RATE,
    x_test, y_test, label='[NonIID]'
)

global_acc_noniid = noniid_round_accs[-1]
print(f'\nNon-IID target model accuracy: {global_acc_noniid:.4f}')

In [ ]:
# ── Step 5: Extract White-Box Features for Non-IID Model ─────────────────────
print('Step 5: Extracting white-box features for Non-IID model...')

rng = np.random.default_rng(RANDOM_SEED + 5)

noniid_m_idx  = rng.choice(noniid_member_indices, size=N_ATTACK, replace=False)
noniid_nm_idx = rng.choice(len(x_test),            size=N_ATTACK, replace=False)

x_nm_members    = x_train[noniid_m_idx]
y_nm_members    = y_train[noniid_m_idx]
x_nm_nonmembers = x_test[noniid_nm_idx]
y_nm_nonmembers = y_test[noniid_nm_idx]

print('Extracting features for Non-IID MEMBERS...')
feat_nm_members = extract_whitebox_features(
    noniid_model, x_nm_members, y_nm_members, label='NI-Members'
)

print('\nExtracting features for Non-IID NON-MEMBERS...')
feat_nm_nonmembers = extract_whitebox_features(
    noniid_model, x_nm_nonmembers, y_nm_nonmembers, label='NI-Non-members'
)

X_attack_noniid = np.vstack([feat_nm_members, feat_nm_nonmembers])
y_attack_noniid = np.array([1]*N_ATTACK + [0]*N_ATTACK)

print(f'Non-IID feature matrix: {X_attack_noniid.shape}')

In [ ]:
# ── Step 6: Train & Evaluate White-Box Attack (Non-IID) ──────────────────────
scaler_noniid   = StandardScaler()
X_ni_scaled     = scaler_noniid.fit_transform(X_attack_noniid)

X_ni_tr, X_ni_te, y_ni_tr, y_ni_te = train_test_split(
    X_ni_scaled, y_attack_noniid,
    test_size=0.3, stratify=y_attack_noniid, random_state=RANDOM_SEED
)

wb_clf_noniid = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
wb_clf_noniid.fit(X_ni_tr, y_ni_tr)
wb_metrics_noniid = evaluate_attack(wb_clf_noniid, X_ni_te, y_ni_te)

loss_auc_noniid = loss_based_auc(
    noniid_model, x_nm_members, y_nm_members, x_nm_nonmembers, y_nm_nonmembers
)

print()
print('═' * 65)
print('WHITE-BOX ATTACK RESULTS — CIFAR-10 (Non-IID, 5/10 classes)')
print('═' * 65)
print(f'  Non-IID model accuracy         : {global_acc_noniid:.4f}')
print()
print(f'  {"Method":<40} {"AUC":>8} {"Recall":>8}')
print(f'  {"-"*56}')
print(f'  {"Loss-based (no training)":<40} {loss_auc_noniid:>8.4f} {"N/A":>8}')
print(f'  {"White-box (gradient features)":<40} {wb_metrics_noniid["auc"]:>8.4f} {wb_metrics_noniid["recall"]:>8.4f}')
print(f'  {"Random chance":<40} {0.5000:>8.4f} {0.5000:>8.4f}')
print('═' * 65)

## Step 7 — White-Box vs Other Methods Comparison

This figure compares the white-box gradient attack against the loss-based
and shadow model attacks under IID and Non-IID conditions.


In [ ]:
# Ranking figure removed — white-box results shown in comparison figure below
print("White-box attack complete. Generating comparison figure...")


In [ ]:

# ── Figure 2: White-box IID vs Non-IID annotated bar ─────────────────────────
metrics_wb  = ['White-box\nAUC', 'White-box\nRecall', 'Loss-based\nAUC']
iid_wb      = [wb_metrics_iid["auc"],    wb_metrics_iid["recall"],    loss_auc_iid]
noniid_wb   = [wb_metrics_noniid["auc"], wb_metrics_noniid["recall"], loss_auc_noniid]
deltas_wb   = [n - i for n, i in zip(noniid_wb, iid_wb)]

x = np.arange(len(metrics_wb)); w = 0.30
fig, ax = plt.subplots(figsize=(11, 6.5))
b1 = ax.bar(x - w/2, iid_wb,    w, color=NAVY, alpha=0.85, label='IID',     edgecolor='white')
b2 = ax.bar(x + w/2, noniid_wb, w, color=RED,  alpha=0.85, label='Non-IID (5/10 classes)', edgecolor='white')

for bar, v in zip(b1, iid_wb):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.008,
            f'{v:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color=NAVY)
for bar, v in zip(b2, noniid_wb):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.008,
            f'{v:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color=RED)

for i, delta in enumerate(deltas_wb):
    y_top = max(iid_wb[i], noniid_wb[i]) + 0.055
    col = RED if delta > 0 else BLUE
    sign = '+' if delta > 0 else ''
    ax.annotate('', xy=(x[i]+w/2, y_top-0.012), xytext=(x[i]-w/2, y_top-0.012),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.8))
    ax.text(x[i], y_top + 0.005, f'{sign}{delta:.3f}',
            ha='center', fontsize=11, color=col, fontweight='bold')

ax.axhline(0.50, **RANDOM_LINE)
ax.set_xticks(x); ax.set_xticklabels(metrics_wb, fontsize=12)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('RQ7: White-Box Gradient Attack — IID vs Non-IID\nNon-IID increases AUC — gradient access amplifies client specialisation',
             fontsize=14, fontweight='bold')
ax.set_ylim(0.44, 0.72)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'whitebox_noniid_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"White-box IID — AUC: {wb_metrics_iid['auc']:.4f}  Recall: {wb_metrics_iid['recall']:.4f}")
print(f"White-box Non-IID — AUC: {wb_metrics_noniid['auc']:.4f}  Recall: {wb_metrics_noniid['recall']:.4f}")
print(f"Highest AUC overall: {max(all_aucs):.4f}")


In [ ]:
# ── Step 9: Final Summary & Save Results ─────────────────────────────────────
print('═' * 65)
print('COMPLETE WHITE-BOX ATTACK SUMMARY — CIFAR-10 FedAvg')
print('═' * 65)
print(f'  {"Method":<42} {"IID AUC":>9} {"NonIID AUC":>12}')
print(f'  {"-"*63}')
print(f'  {"Random chance":<42} {0.500:>9.4f} {0.500:>12.4f}')
print(f'  {"Black-box LR (unrealistic)":<42} {bb_metrics_iid["auc"]:>9.4f} {"—":>12}')
print(f'  {"Shadow model (CIFAR-100)":<42} {shadow_auc_iid:>9.4f} {shadow_auc_noniid:>12.4f}')
print(f'  {"Loss-based (no attacker training)":<42} {loss_auc_iid:>9.4f} {loss_auc_noniid:>12.4f}')
print(f'  {"White-box (gradient features)":<42} {wb_metrics_iid["auc"]:>9.4f} {wb_metrics_noniid["auc"]:>12.4f}')
print('═' * 65)
print()
print(f'  White-box gain over loss-based (IID):    {wb_metrics_iid["auc"]-loss_auc_iid:+.4f}')
print(f'  White-box gain over shadow model (IID):  {wb_metrics_iid["auc"]-shadow_auc_iid:+.4f}')
print(f'  Non-IID increase (white-box):            {wb_metrics_noniid["auc"]-wb_metrics_iid["auc"]:+.4f}')

results = {
    'target_global_acc_iid':        float(global_acc_iid),
    'target_global_acc_noniid':     float(global_acc_noniid),
    'blackbox_lr_auc_iid':          float(bb_metrics_iid['auc']),
    'blackbox_lr_recall_iid':       float(bb_metrics_iid['recall']),
    'shadow_auc_iid':               float(shadow_auc_iid),
    'shadow_auc_noniid':            float(shadow_auc_noniid),
    'loss_auc_iid':                 float(loss_auc_iid),
    'loss_auc_noniid':              float(loss_auc_noniid),
    'whitebox_auc_iid':             float(wb_metrics_iid['auc']),
    'whitebox_recall_iid':          float(wb_metrics_iid['recall']),
    'whitebox_auc_noniid':          float(wb_metrics_noniid['auc']),
    'whitebox_recall_noniid':       float(wb_metrics_noniid['recall']),
    'whitebox_gain_over_loss_iid':  float(wb_metrics_iid['auc'] - loss_auc_iid),
    'whitebox_gain_over_shadow_iid':float(wb_metrics_iid['auc'] - shadow_auc_iid),
    'whitebox_noniid_delta':        float(wb_metrics_noniid['auc'] - wb_metrics_iid['auc']),
}

with open(os.path.join(OUTPUT_DIR, 'whitebox_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('\nResults saved to whitebox_results.json')

In [ ]:
# ── Download outputs ──────────────────────────────────────────────────────────
from google.colab import files
files.download(os.path.join(OUTPUT_DIR, 'whitebox_noniid_comparison.png'))
files.download(os.path.join(OUTPUT_DIR, 'whitebox_results.json'))
print('Done!')

## Summary Heatmap

This final figure consolidates all results from all seven notebooks into a single matrix. Each row is an attack method. Each column is an experimental condition. Darker colour means stronger privacy leakage. Grey cells were not evaluated under that condition.

In [ ]:

# ── Figure: Summary Heatmap — All results in one view ────────────────────────
# This figure consolidates every attack method and condition into a single
# readable matrix. Rows = attack methods. Columns = experimental conditions.
# Darker colour = stronger privacy leakage.

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'figure.dpi': 150, 'savefig.dpi': 150,
    'savefig.facecolor': 'white',
})

row_labels = [
    'Black-box LR
(unrealistic)',
    'Threshold attack',
    'Loss-based attack',
    'Shadow model',
    'White-box gradient',
]
col_labels = [
    'CIFAR-10
IID AUC',
    'CIFAR-10
Non-IID AUC',
    'CIFAR-100
IID AUC',
    'Member
Recall (IID)',
]

# Fill with actual computed values from this notebook and previous results
data = np.array([
    [bb_metrics_iid["auc"],       np.nan,                   0.516,   bb_metrics_iid["recall"]  ],
    [0.536,                        np.nan,                   0.541,   np.nan                    ],
    [loss_auc_iid,                 loss_auc_noniid,          0.618,   np.nan                    ],
    [shadow_auc_iid,               shadow_auc_noniid,        np.nan,  shadow_recall_iid         ],
    [wb_metrics_iid["auc"],        wb_metrics_noniid["auc"], np.nan,  wb_metrics_iid["recall"]  ],
])

fig, ax = plt.subplots(figsize=(12, 6.5))

# Custom colormap — grey for not-evaluated cells, yellow-orange-red for values
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad('#F1F5F9')   # light grey for NaN

masked = np.ma.masked_invalid(data)
im = ax.imshow(masked, cmap=cmap, aspect='auto', vmin=0.47, vmax=0.65)

# Annotate every cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        if not np.isnan(val):
            # White text on dark cells, dark text on light cells
            text_color = 'white' if val > 0.58 else '#1E293B'
            weight = 'bold' if val >= 0.60 else 'normal'
            ax.text(j, i, f'{val:.3f}',
                    ha='center', va='center',
                    fontsize=13, color=text_color, fontweight=weight)
        else:
            ax.text(j, i, '—',
                    ha='center', va='center',
                    fontsize=13, color='#CBD5E1')

# Highlight the single highest value with a border
max_idx = np.unravel_index(np.nanargmax(data), data.shape)
ax.add_patch(plt.Rectangle(
    (max_idx[1] - 0.5, max_idx[0] - 0.5), 1, 1,
    fill=False, edgecolor='#0D1B2A', linewidth=3
))

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=12)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Attack AUC / Member Recall', fontsize=11)
cbar.ax.tick_params(labelsize=10)

ax.set_title(
    'Summary: All Attack Methods × All Experimental Conditions\n'
    'Darker = stronger privacy leakage  |  Grey (—) = not evaluated in this condition',
    fontsize=14, fontweight='bold', pad=14
)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'summary_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Summary heatmap saved.")
print(f"Highest value in heatmap: {np.nanmax(data):.3f} at row '{row_labels[max_idx[0]]}', col '{col_labels[max_idx[1]]}'")
